In [ ]:
Api_key =  "sk_a5ba8a37b338c304ad00df23c7dbb876f86be2e69c0f19ec"

In [ ]:
import os
import json
import requests
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
ELEVENLABS_API_KEY = "sk_a5ba8a37b338c304ad00df23c7dbb876f86be2e69c0f19ec"
VOICE_ID           = "JBFqnCBsd6RMkjVDRZzb"
MODEL_ID           = "eleven_turbo_v2_5"
OUTPUT_DIR         = "changed_frames"
AUDIO_DIR          = "audio_clips"
SCRIPT_PATH        = os.path.join(OUTPUT_DIR, "gemini_script.json")

# Parallelism config
MAX_WORKERS        = 4    # ElevenLabs free tier: keep at 2-3
                          # ElevenLabs paid tier: safe up to 5-6
MAX_RETRIES        = 3

os.makedirs(AUDIO_DIR, exist_ok=True)

# Thread-safe print lock
print_lock = threading.Lock()
def tprint(msg):
    with print_lock:
        print(msg)

# ─────────────────────────────────────────
# STEP 1: LOAD SCRIPT
# ─────────────────────────────────────────
with open(SCRIPT_PATH) as f:
    script = json.load(f)

print(f"🎙️  Total narrations to convert: {len(script)}")

# ─────────────────────────────────────────
# STEP 2: GENERATE AUDIO (single request)
# ─────────────────────────────────────────
def generate_audio(entry, index, total):
    frame     = entry.get("frame", f"frame_{index}")
    narration = entry.get("narration", "").strip()
    position  = entry.get("position", "middle")

    # Carry over timestamps from gemini_script.json
    timestamp_s  = entry.get("timestamp_s", 0)
    srt_time     = entry.get("srt_time", "00:00:00,000")
    frame_number = entry.get("frame_number", 0)

    if not narration:
        tprint(f"  ⚠️  [{index+1}/{total}] Skipping empty narration: {frame}")
        return None

    audio_filename = Path(frame).stem + ".mp3"
    audio_path     = os.path.join(AUDIO_DIR, audio_filename)

    # Skip if already generated (resume support)
    if os.path.exists(audio_path) and os.path.getsize(audio_path) > 0:
        tprint(f"  ⏭️  [{index+1}/{total}] Already exists, skipping: {audio_filename}")
        return {
            "frame":        frame,
            "audio":        audio_filename,
            "narration":    narration,
            "position":     position,
            "audio_path":   audio_path,
            "timestamp_s":  timestamp_s,
            "srt_time":     srt_time,
            "frame_number": frame_number,
            "index":        index,
        }

    url     = f"https://api.elevenlabs.io/v1/text-to-speech/{VOICE_ID}"
    headers = {
        "xi-api-key":   ELEVENLABS_API_KEY,
        "Content-Type": "application/json"
    }
    payload = {
        "text":     narration,
        "model_id": MODEL_ID,
        "voice_settings": {
            "stability":         0.5,
            "similarity_boost":  0.75,
            "style":             0.3,
            "use_speaker_boost": True
        }
    }

    for attempt in range(MAX_RETRIES):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=30)

            if response.status_code == 200:
                with open(audio_path, "wb") as f:
                    f.write(response.content)
                size_kb = os.path.getsize(audio_path) / 1024
                tprint(f"  ✅ [{index+1}/{total}] {audio_filename} ({round(size_kb)}KB) ← {position}")
                return {
                    "frame":        frame,
                    "audio":        audio_filename,
                    "narration":    narration,
                    "position":     position,
                    "audio_path":   audio_path,
                    "timestamp_s":  timestamp_s,
                    "srt_time":     srt_time,
                    "frame_number": frame_number,
                    "index":        index,
                }

            elif response.status_code == 429:
                wait = 2 ** (attempt + 1)   # 2s, 4s, 8s
                tprint(f"  ⏳ [{index+1}/{total}] Rate limited, retrying in {wait}s...")
                time.sleep(wait)

            else:
                tprint(f"  ❌ [{index+1}/{total}] Error {response.status_code}: {response.text[:100]}")
                return None

        except requests.exceptions.Timeout:
            tprint(f"  ⏳ [{index+1}/{total}] Timeout on attempt {attempt+1}, retrying...")
            time.sleep(2 ** attempt)

        except Exception as e:
            tprint(f"  ❌ [{index+1}/{total}] Unexpected error: {e}")
            return None

    tprint(f"  ❌ [{index+1}/{total}] Failed after {MAX_RETRIES} retries: {frame}")
    return None

# ─────────────────────────────────────────
# STEP 3: PARALLEL EXECUTION
# ─────────────────────────────────────────
print(f"\n⚡ Generating audio in parallel (workers={MAX_WORKERS})...")
start_time = time.time()

results = [None] * len(script)  # preserve order

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(generate_audio, entry, i, len(script)): i
        for i, entry in enumerate(script)
        if entry.get("narration", "").strip()
    }

    for future in as_completed(futures):
        idx         = futures[future]
        results[idx] = future.result()

# Filter failed/skipped
audio_meta = sorted(
    [r for r in results if r is not None],
    key=lambda x: x["index"]   # restore original order
)

elapsed = round(time.time() - start_time, 1)

# ─────────────────────────────────────────
# STEP 4: SAVE AUDIO METADATA
# ─────────────────────────────────────────
# Clean up index field before saving
for entry in audio_meta:
    entry.pop("index", None)

audio_meta_path = os.path.join(AUDIO_DIR, "audio_meta.json")
with open(audio_meta_path, "w") as f:
    json.dump(audio_meta, f, indent=2)

# ─────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────
print(f"\n{'─'*55}")
print(f"🎙️  AUDIO GENERATION COMPLETE")
print(f"{'─'*55}")
print(f"Total narrations:    {len(script)}")
print(f"Audio files saved:   {len(audio_meta)}")
print(f"Time taken:          {elapsed}s")
print(f"Output folder:       {AUDIO_DIR}/")
print(f"Metadata saved:      {audio_meta_path}")
print(f"{'─'*55}")
for a in audio_meta:
    print(f"  🔊 {a['audio']}  ⏱️  {a['timestamp_s']}s  ← {a['position']}")

🎙️  Total narrations to convert: 17

[1/17] frame_01.png (opening)
  📝 Welcome to VisionCurator, your platform for intelligent dataset management. In t...
  ✅ Saved: frame_01.mp3

[2/17] frame_02.png (early)
  📝 Here, we see our 'dogs' dataset, already processed and analyzed. VisionCurator a...
  ✅ Saved: frame_02.mp3

[3/17] frame_03.png (early)
  📝 Clicking 'Explore' reveals the full image gallery, allowing you to visually insp...
  ✅ Saved: frame_03.mp3

[4/17] frame_04.png (early)
  📝 This comprehensive view ensures you understand the breadth and depth of your dat...
  ✅ Saved: frame_04.mp3

[5/17] frame_05.png (early)
  📝 VisionCurator organizes your data into meaningful segments. Let's look at the cl...
  ✅ Saved: frame_05.mp3

[6/17] frame_06.png (early)
  📝 Here, we see distinct clusters, like Cluster #0 containing larger dogs and Clust...
  ✅ Saved: frame_06.mp3

[7/17] frame_07.png (middle)
  📝 Scrolling down, we can observe more clusters, such as Cluster #2, each represent..